In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

In [5]:
df = pd.read_csv("cleaned_ahemadabad.csv")

In [6]:
print("Shape:", df.shape)

Shape: (5839, 6)


In [7]:
print(df.head())

   price  price_sqft  bhk                         location  total_sqft  \
0   16.0        3765    1  Vejalpur, Jivraj park Ahmedabad         425   
1   24.0        4800    1  Vejalpur, Jivraj park Ahmedabad         500   
2   40.0        5487    1  Vejalpur, Jivraj park Ahmedabad         729   
3   16.0        3200    1  Vejalpur, Jivraj park Ahmedabad         500   
4   20.0        5000    1  Vejalpur, Jivraj park Ahmedabad         400   

   floor_num  
0          2  
1          3  
2          1  
3          2  
4          2  


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5839 entries, 0 to 5838
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   price       5839 non-null   float64
 1   price_sqft  5839 non-null   int64  
 2   bhk         5839 non-null   int64  
 3   location    5839 non-null   object 
 4   total_sqft  5839 non-null   int64  
 5   floor_num   5839 non-null   int64  
dtypes: float64(1), int64(4), object(1)
memory usage: 273.8+ KB


In [9]:
print(df.describe(include="all"))

              price    price_sqft          bhk location     total_sqft  \
count   5839.000000   5839.000000  5839.000000     5839    5839.000000   
unique          NaN           NaN          NaN      191            NaN   
top             NaN           NaN          NaN    other            NaN   
freq            NaN           NaN          NaN     2888            NaN   
mean      23.976982   6403.748930     3.169892      NaN    2249.630074   
std       28.338212   2430.724144     1.081112      NaN    9908.317812   
min        1.000000   1550.000000     1.000000      NaN     300.000000   
25%        2.680000   4583.000000     2.000000      NaN    1080.000000   
50%        5.220000   6042.000000     3.000000      NaN    1710.000000   
75%       47.000000   8100.000000     4.000000      NaN    2700.000000   
max      100.000000  54386.000000     9.000000      NaN  750000.000000   

          floor_num  
count   5839.000000  
unique          NaN  
top             NaN  
freq            NaN  
m

In [18]:
print("Missing values:\n", df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())
print(df.dtypes)

df = df.drop_duplicates().reset_index(drop=True)
df = df.dropna(subset=["price", "total_sqft", "location"])

Missing values:
 price             0
price_sqft        0
bhk               0
location          0
total_sqft        0
floor_num         0
price_category    0
dtype: int64
Duplicate rows: 288
price              float64
price_sqft           int64
bhk                  int64
location            object
total_sqft           int64
floor_num            int64
price_category    category
dtype: object


In [19]:
# price_sqft * total_sqft / 1e5 = expected price in lakhs
df["expected_lakh"] = df["price_sqft"] * df["total_sqft"] / 1e5
df["ratio"] = df["expected_lakh"] / df["price"]

# If expected is far larger than price, the price is probably in crores
df["unit_guess"] = np.where(df["ratio"] > 20, "Crore", "Lakh")
print(df["unit_guess"].value_counts())
print(df[df["unit_guess"] == "Crore"].head())

# Convert everything to lakhs
df["price_lakh"] = np.where(df["unit_guess"] == "Crore", df["price"] * 100, df["price"])

unit_guess
Crore    3311
Lakh     2240
Name: count, dtype: int64
    price  price_sqft  bhk                           location  total_sqft  \
21   4.25        9463    5  Sankalp Grace 3, Shilaj Ahmedabad        2694   
22   4.21        7478    5  Sankalp Grace 3, Shilaj Ahmedabad        3400   
23   4.50       10000    5  Sankalp Grace 3, Shilaj Ahmedabad        2500   
24   4.25        9548    5  Sankalp Grace 3, Shilaj Ahmedabad        3200   
25   4.29        7001    5  Sankalp Grace 3, Shilaj Ahmedabad        4729   

    floor_num price_category  expected_lakh      ratio unit_guess  
21         12      Mid-range      254.93322  59.984287      Crore  
22         12      Mid-range      254.25200  60.392399      Crore  
23          5      Mid-range      250.00000  55.555556      Crore  
24         12      Mid-range      305.53600  71.890824      Crore  
25         12      Mid-range      331.07729  77.174193      Crore  


In [20]:
df["location"] = df["location"].str.strip()
df["area"] = (df["location"]
              .str.replace("Ahmedabad", "", regex=False)
              .str.split(",").str[-1]       # last part is the locality
              .str.strip()
              .str.title())

print(df["area"].nunique(), "unique areas")
print(df["area"].value_counts().head(10))

76 unique areas
area
Other                 2733
Shela                  335
Ambli                  270
Bodakdev               232
Thaltej                167
Bopal                  123
Iscon Ambli Road       121
Shilaj                 119
Vaishnodevi Circle     111
Satellite               97
Name: count, dtype: int64


In [21]:
for c in ["bhk", "floor_num"]:
    df[c] = df[c].astype(int)

# Remove impossible values
df = df[(df["total_sqft"] > 0) & (df["price_lakh"] > 0) & (df["price_sqft"] > 0)]
print("Rows after removing invalid values:", len(df))

Rows after removing invalid values: 5551


In [22]:
def iqr_bounds(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

px.box(df, y="price_sqft", title="Price per Sqft (before outlier removal)").show()

lo, hi = iqr_bounds(df["price_sqft"])
print("Price/sqft bounds:", lo, hi)
print("Outliers:", ((df["price_sqft"] < lo) | (df["price_sqft"] > hi)).sum())

df_clean = df[(df["price_sqft"] >= lo) & (df["price_sqft"] <= hi)].copy()
print("Rows before:", len(df), "after:", len(df_clean))

Price/sqft bounds: -723.75 13434.25
Outliers: 28
Rows before: 5551 after: 5523


In [23]:
print(df_clean.groupby("bhk")[["price_lakh", "price_sqft", "total_sqft"]].mean().round(1))

area_stats = (df_clean.groupby("area")
              .agg(listings=("price_lakh", "count"),
                   avg_price_lakh=("price_lakh", "mean"),
                   avg_price_sqft=("price_sqft", "mean"))
              .round(1)
              .sort_values("avg_price_sqft", ascending=False))
print(area_stats.head(10))

print(df_clean[["price_lakh", "price_sqft", "bhk", "total_sqft", "floor_num"]].corr().round(2))

     price_lakh  price_sqft  total_sqft
bhk                                    
1          25.5      4063.9       596.6
2          50.4      4512.6      1004.6
3         116.1      5893.3      1592.0
4         334.7      7884.3      2947.1
5         534.1      8317.2      6015.2
6         721.6      8791.6      5097.4
7         752.0      7500.0      5520.0
8         540.0      7194.0      7506.0
9         510.0      7155.0      4276.0
                           listings  avg_price_lakh  avg_price_sqft
area                                                               
Iscon Ambli Road                121           559.4          9811.5
Rajpath Rangoli Road              7           588.7          9099.3
Ambli Bopal Road                 96           535.1          9080.4
Ambli Road                        6           422.3          8983.3
Bodakdev                        232           452.7          8881.7
Chharodi                         29           264.8          8652.6
Sindhu Bhavan Ma

In [25]:
px.histogram(df_clean, x="price_lakh", nbins=30,
             title="Price Distribution (Lakhs)").show()

In [26]:
px.bar(df_clean.groupby("bhk")["price_lakh"].mean().reset_index(),
       x="bhk", y="price_lakh", title="Average Price by BHK").show()

In [27]:
px.bar(area_stats.head(10).reset_index(), x="area", y="avg_price_sqft",
       title="Top 10 Areas by Price per Sqft").show()

In [28]:
px.scatter(df_clean, x="total_sqft", y="price_lakh", color="bhk",
           title="Area vs Price").show()

In [29]:
px.imshow(df_clean[["price_lakh", "price_sqft", "bhk", "total_sqft", "floor_num"]].corr(),
          text_auto=".2f", title="Correlation Heatmap").show()